In [238]:
import torch
import torch.nn as nn
import random

In [239]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

    def forward(self, X):
        starting_matrix = torch.zeros((X.shape[-2], X.shape[-1]))

        encoded_matrix = []

        position = 0

        for row in starting_matrix:

            final_row_encode = torch.clone(row)

            for i in range(0, len(row), 2):

                div_term = 10000 ** (i / self.d_model)

                final_row_encode[i] = torch.sin(torch.tensor(position / div_term))
                final_row_encode[i + 1] = torch.cos(torch.tensor(position / div_term))

            position += 1

            encoded_matrix.append(final_row_encode)

        encoded_matrix = torch.stack(encoded_matrix)

        return X + encoded_matrix

In [240]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.softmax = nn.Softmax(dim=-1)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, X):
        Q = self.q_proj(X)
        V = self.v_proj(X)
        K = self.k_proj(X)

        dim_model_single_head = self.d_model // self.num_heads

        list_Q = []
        list_V = []
        list_K = []

        for i in range(0, Q.shape[-1], dim_model_single_head):
            list_Q.append(Q[..., i:i+dim_model_single_head])
            list_K.append(K[..., i:i+dim_model_single_head])
            list_V.append(V[..., i:i+dim_model_single_head])

        slices_with_attention = []

        for i in range(len(list_Q)):
            initial = torch.matmul(list_Q[i], list_K[i].transpose(-2, -1))

            initial_mask = torch.triu(torch.ones(initial.shape[-2], initial.shape[-1]), diagonal=1).bool()

            initial = initial.masked_fill(initial_mask, float("-inf"))

            normalized = self.softmax(torch.div(initial, torch.sqrt(torch.tensor(dim_model_single_head, dtype=torch.float32))))

            slices_with_attention.append(torch.matmul(normalized, list_V[i]))

        concatenated = torch.concat(slices_with_attention, dim=-1)

        return self.out_proj(concatenated)

In [241]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj_hidden_layer = nn.Linear(d_model, d_model * 4)
        self.proj_outer_layer = nn.Linear(d_model * 4, d_model)
        self.relu = nn.ReLU()

    def forward(self, X):
        hidden_layer = self.relu(self.proj_hidden_layer(X))

        return self.proj_outer_layer(hidden_layer)

In [242]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, X):
        matrix_mean = torch.mean(X, dim=-1, keepdims=True)
        matrix_standard_deviation = torch.std(X, dim=-1, keepdims=True)
        
        main_calculation = torch.div(torch.sub(X, matrix_mean), torch.add(matrix_standard_deviation, self.eps))

        scaled_calc = torch.mul(self.gamma, main_calculation)

        return torch.add(scaled_calc, self.beta)

In [243]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
        self.first_layer_norm = LayerNorm(d_model)
        self.second_layer_norm = LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model)

    def forward(self, X):
        multi_head = self.multi_head_attention(X)

        residual_1 = multi_head + X
        layer_norm_1 = self.first_layer_norm(residual_1)

        feed_forward = self.feed_forward(layer_norm_1)

        residual_2 = feed_forward + layer_norm_1
        layer_norm_2 = self.second_layer_norm(residual_2)

        return layer_norm_2

In [244]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks):
        super().__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.proj_position_layer = PositionalEncoding(d_model=d_model)
        self.sequential = nn.Sequential(*[TransformerBlock(d_model=d_model, num_heads=num_heads) for _ in range(num_blocks)])
        self.proj_linear = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        embeded = self.embedding_layer(token_ids)

        embeded_with_position_encoding = self.proj_position_layer(embeded)

        transformer_blocks = self.sequential(embeded_with_position_encoding)

        return self.proj_linear(transformer_blocks)    

In [245]:
# torch.manual_seed(42)
# model = Transformer(vocab_size=100, d_model=4, num_heads=2, num_blocks=2)

# token_ids = torch.tensor([1, 5, 17])
# output = model(token_ids)
# print(output.shape)
# print(output)

In [246]:
with open("./on_war.txt", "r", encoding="utf-8") as f:
    content = f.read()

vocabulary = sorted(set(content))

print("Vocabulary length: " + str(len(vocabulary)))

map_to_integer_id = {}
map_to_character = {}

for i in range(len(vocabulary)):
    map_to_integer_id[vocabulary[i]] = i
    map_to_character[i] = vocabulary[i]

def encode(string):
    list_of_integer_ids = []

    for char in string:
        list_of_integer_ids.append(map_to_integer_id[char])

    return torch.tensor(list_of_integer_ids, dtype=torch.long)

def decode(list_of_integer_ids):
    string = ""

    for id in list_of_integer_ids:
        string += map_to_character[int(id)]

    return string

print(encode("war"))
print(decode(encode("war")))


Vocabulary length: 105
tensor([84, 62, 79])
war


In [247]:
def get_batch(data, batch_size, block_size):
    data_length = len(data)
    random_positions = [random.randint(0, len(data) - 1 - block_size) for _ in range(batch_size)]

    input_slices = []
    target_slices = []

    for position in random_positions:
        input_slices.append(data[position:position+block_size])
        target_slices.append(data[position+1:position+block_size+1])

    return torch.stack(input_slices), torch.stack(target_slices)


In [248]:
data = torch.as_tensor(encode(content), dtype=torch.long)

inputs, targets = get_batch(data, batch_size=4, block_size=8)
print(inputs.shape)   # torch.Size([4, 8])
print(targets.shape)  # torch.Size([4, 8])
print(inputs)
print(targets)
print(decode(inputs[0].tolist())) 

torch.Size([4, 8])
torch.Size([4, 8])
tensor([[69, 62, 77, 80,  2, 81, 69, 70],
        [75,  2, 81, 69, 62, 81,  2, 69],
        [74, 86, 14,  2, 63, 82, 81,  2],
        [76, 67,  2, 69, 66, 70, 68, 69]])
tensor([[62, 77, 80,  2, 81, 69, 70, 75],
        [ 2, 81, 69, 62, 81,  2, 69, 70],
        [86, 14,  2, 63, 82, 81,  2, 70],
        [67,  2, 69, 66, 70, 68, 69, 81]])
haps thi


In [253]:
batch_size=32
block_size=128
d_model=128
num_heads=4
num_blocks=4
lr=3e-4
num_steps=3200

loss_function = nn.CrossEntropyLoss()
transformer = Transformer(vocab_size=len(vocabulary), d_model=d_model, num_heads=num_heads, num_blocks=num_blocks)
optimizer = torch.optim.AdamW(params=transformer.parameters(), lr=lr)

for i in range(num_steps):
        inputs, targets = get_batch(data, batch_size=batch_size, block_size=block_size)

        forward_pass = transformer(inputs)

        targets_loss = targets.reshape(-1)
        forward_pass_loss = forward_pass.reshape(-1, forward_pass.size(-1))
        
        loss = loss_function(forward_pass_loss, targets_loss)
        
        if i % 5 == 0:
                print(f"step {i}: loss = {loss.item():.4f}")

        loss.backward()

        optimizer.step()

        optimizer.zero_grad()

step 0: loss = 4.8732
step 5: loss = 3.6921
step 10: loss = 3.3472
step 15: loss = 3.2350
step 20: loss = 3.1396
step 25: loss = 2.9472
step 30: loss = 2.8361
step 35: loss = 2.8204
step 40: loss = 2.7890
step 45: loss = 2.6503
step 50: loss = 2.6179
step 55: loss = 2.6372
step 60: loss = 2.5631
step 65: loss = 2.6157
step 70: loss = 2.6096
step 75: loss = 2.5394
step 80: loss = 2.5303
step 85: loss = 2.5288
step 90: loss = 2.4846
step 95: loss = 2.5391
step 100: loss = 2.4820
step 105: loss = 2.5259
step 110: loss = 2.4960
step 115: loss = 2.4657
step 120: loss = 2.4958
step 125: loss = 2.4106
step 130: loss = 2.4293
step 135: loss = 2.4551
step 140: loss = 2.4621
step 145: loss = 2.5227
step 150: loss = 2.4614
step 155: loss = 2.4426
step 160: loss = 2.4540
step 165: loss = 2.4195
step 170: loss = 2.5129
step 175: loss = 2.3929
step 180: loss = 2.4095
step 185: loss = 2.4209
step 190: loss = 2.4023
step 195: loss = 2.4039
step 200: loss = 2.4184
step 205: loss = 2.4122
step 210: loss

In [275]:
def generate(model, prompt_text, max_new_tokens): 
    encoded = encode(prompt_text)
    new_tokens = 0

    while new_tokens != max_new_tokens:
        prediciton = model(encoded.unsqueeze(0))

        logit_at_last_position = prediciton[0][-1]
        
        next_char_int_id = torch.argmax(logit_at_last_position)

        probs = torch.softmax(logit_at_last_position, dim=-1)
        
        # top_10 = torch.topk(probs, 10)

        # top_10_indexes = top_10.indices

        # top10 = torch.softmax(top_10.values, dim=-1)

        # next_char_int_id = top_10_indexes[torch.multinomial(top_10, num_samples=1)]
        
        character = map_to_character[int(next_char_int_id)]

        new_tokens += 1
        
        # encoded = torch.cat((encoded, next_char_int_id), dim=0)
        encoded = torch.cat((encoded, torch.tensor([next_char_int_id])), dim=0)

        if (len(encoded) > block_size):
            encoded = encoded[1:]

    return decode(encoded)
        
print(generate(transformer, "There is", block_size))        

 not a completely in the consideration of the enemy’s
army, therefore, the more of the completely of the enemy’s army in
the con
